In [ ]:
# ── CELL 1: Cài vLLM ─────────────────────────────────────────────
!pip uninstall -q vllm -y
!pip install -q vllm --extra-index-url https://wheels.vllm.ai/nightly
print('✅ Cài vLLM xong!')

In [ ]:
# ── CELL 2: Khởi động vLLM LLM Server + localtunnel ──────────────
import os, re, time, subprocess, threading, requests

MODEL_NAME = os.environ.get('MODEL_NAME', 'Qwen/Qwen2.5-14B-Instruct')
VLLM_PORT  = int(os.environ.get('VLLM_PORT', 8082))
SUBDOMAIN  = os.environ.get('SUBDOMAIN', 'vks-llm-hvks')
FIXED_URL  = f'https://{SUBDOMAIN}.loca.lt'


def start_vllm():
    cmd = [
        'vllm', 'serve', MODEL_NAME,
        '--port',                    str(VLLM_PORT),
        '--host',                    '0.0.0.0',
        '--dtype',                   'bfloat16',
        '--max-model-len',           '8192',
        '--max-num-seqs',            '16',
        '--gpu-memory-utilization',  '0.9',
        '--enable-prefix-caching',
        '--trust-remote-code',
    ]
    def _run():
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='')
    threading.Thread(target=_run, daemon=True).start()


def wait_for_vllm(timeout=1200):
    url = f'http://localhost:{VLLM_PORT}/v1/models'
    t0  = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=5).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False


def install_localtunnel():
    if subprocess.run('which lt', shell=True, capture_output=True).returncode == 0:
        return
    subprocess.run('npm install -g localtunnel -q', shell=True, timeout=120)


def start_tunnel():
    log_file = 'tunnel_llm.log'
    subprocess.run("pkill -9 -f 'lt --port' || true", shell=True, timeout=5)
    time.sleep(2)
    try:
        os.remove(log_file)
    except FileNotFoundError:
        pass
    subprocess.Popen(
        f'lt --port {VLLM_PORT} --subdomain {SUBDOMAIN} > {log_file} 2>&1 &',
        shell=True,
    )
    for _ in range(40):
        time.sleep(1)
        try:
            with open(log_file) as f:
                log = f.read()
            m = re.search(r'https://[\w-]+\.loca\.lt', log)
            if m:
                return m.group(0)
        except FileNotFoundError:
            pass
    return None


# ── Main ──────────────────────────────────────────────────────────
print(f'[1/4] start vLLM on port {VLLM_PORT} (model={MODEL_NAME}) ...', flush=True)
start_vllm()

print('[2/4] wait for vLLM to become ready ...', flush=True)
if not wait_for_vllm():
    print('❌ vLLM không khởi động được trong 20 phút.')
else:
    print('      ✅ vLLM ready')

    print('[3/4] install localtunnel if missing ...', flush=True)
    install_localtunnel()

    print(f'[4/4] expose tunnel subdomain={SUBDOMAIN} ...', flush=True)
    url = None
    for attempt in range(1, 4):
        url = start_tunnel()
        if url == FIXED_URL:
            break
        print(f'      attempt {attempt}/3 got {url!r}, retry ...', flush=True)

    if not url:
        print(f"❌ Không chiếm được subdomain '{SUBDOMAIN}'. Đổi SUBDOMAIN rồi thử lại.")
    else:
        try:
            pw = requests.get('https://loca.lt/mytunnelpassword', timeout=10).text.strip()
        except Exception:
            pw = '<lấy tay tại https://loca.lt/mytunnelpassword>'

        print('\n' + '='*60)
        print(f'✅ vLLM LLM tunnel: {url}')
        print(f'🔑 Tunnel password (IP Colab): {pw}')
        print(f'📋 Endpoint OpenAI-compatible:')
        print(f'   GET  {url}/v1/models')
        print(f'   POST {url}/v1/chat/completions')
        print('='*60)
        print(f'\n→ Trên máy local, set env rồi chạy:')
        print(f'   VLLM_LLM_URL={url} python src/core/pipeline.py')

In [ ]:
# ── CELL 3: Keep-alive (giữ Colab không disconnect) ───────────────
from IPython.display import display, Javascript
display(Javascript(r'''
(function(){
  if (window.__llmKeepAlive) return;
  window.__llmKeepAlive = true;
  setInterval(function(){
    try {
      document.querySelectorAll('colab-connect-button').forEach(b => {
        const s = b.shadowRoot;
        if (s) { const btn = s.querySelector('#connect'); if (btn) btn.click(); }
      });
      document.dispatchEvent(new MouseEvent('mousemove', {clientX:1, clientY:1, bubbles:true}));
    } catch(e){}
  }, 45*1000);
})();
'''))